<a href="https://colab.research.google.com/github/DurDar/Creaciones-Infinitas-Tools/blob/main/Modulo_Api_TiendaNube.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛠 MANUAL DE ESTILO Y LINEAMIENTOS TÉCNICOS
> **Instrucción para IA:** Actúa como un experto en Data Science y Automatización. Al generar o refactorizar código en este cuaderno, debes respetar estrictamente los siguientes lineamientos:

### 1. Gestión de Archivos y Rutas
* **Ruta Base:** Utilizar siempre la variable `BASE_PATH` definida en la configuración inicial. Queda prohibido el uso de rutas absolutas fijas (ej. `/content/drive/...`).
* **Consistencia:** Usar `os.path.join()` para construir rutas, garantizando compatibilidad entre sistemas.

### 2. Estándares de Programación (Clean Code)
* **Feedback Visual:** Todo bucle que procese colecciones de datos (listas, dataframes, archivos) debe implementar `tqdm` para informar el progreso.
* **Silenciamiento:** Usar `%%capture` para instalaciones de librerías para mantener el cuaderno limpio.
* **Visualización:** Presentar siempre los DataFrames mediante `google.colab.data_table` para facilitar la inspección.

### 3. Lógica de Dominio y Validación
* **Entidades Maestras:** Identificar la variable clave del proyecto (ID, SKU, Timestamp, etc.) y asegurar que sea el eje de las transformaciones.
* **Reglas de Negocio Parametrizadas:** El código debe contemplar excepciones de dominio (ej. omitir procesos que no aplican a ciertos registros según sus atributos).
* **Integridad:** Validar la existencia de archivos y la presencia de datos nulos antes de ejecutar cálculos críticos, informando cualquier omisión.

## ⚙️ Configuración Inicial del Entorno
Esta celda se encarga de preparar el entorno de trabajo. Monta Google Drive para acceder a archivos, define la ruta base del proyecto (`BASE_PATH`) y configura la visualización interactiva de DataFrames con `google.colab.data_table`.

In [ ]:
# @title ⚙️ Configuración Inicial de Entorno { display-mode: "form" }
import os
from google.colab import drive, data_table
from tqdm.auto import tqdm

# Habilitar tablas interactivas
data_table.enable_list_custom_formatter()

def preparar_entorno(nombre_carpeta_proyecto="PROYECTO_ACTUAL"):
    """Configura el acceso a Drive y establece la ruta maestra."""
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')

    global BASE_PATH
    # Cambiá "PROYECTO_ACTUAL" por el nombre de tu carpeta en Drive
    BASE_PATH = f'/content/drive/MyDrive/{nombre_carpeta_proyecto}'

    if os.path.exists(BASE_PATH):
        print(f"✅ Entorno vinculado con éxito.")
        print(f"📂 Ruta Maestra: {BASE_PATH}")
    else:
        print(f"⚠️ Alerta: No se encontró la carpeta '{nombre_carpeta_proyecto}' en Drive.")

# Llamada inicial (podés cambiar el nombre de la carpeta aquí)
preparar_entorno("Creaciones_Infinitas")

## 🔑 Carga Segura de Credenciales y Configuración de API
Aquí se cargan las credenciales de la API (SHPAT y STORE_ID) de forma segura desde los 'Secrets' de Colab. También se configuran los `headers` globales para las solicitudes HTTP y la `url_base` de la API de Tienda Nube.

In [ ]:
from google.colab import userdata
import requests
import time
from tqdm.auto import tqdm # Ensure tqdm is imported for general use

# Recuperamos los secretos de la "llave" de Colab
# Asegurate de que 'TN_TOKEN' y 'TN_STORE_ID' esten configurados en Secrets (icono de llave 🔑)
try:
    SHPAT = userdata.get('TN_TOKEN')
    STORE_ID = userdata.get('TN_STORE_ID')
    print("✅ Credenciales cargadas de forma segura (SHPAT y STORE_ID).")
except Exception as e:
    print(f"❌ Error al cargar credenciales: {e}")
    print("Asegurate de que los 'Secrets' con nombres 'TN_TOKEN' y 'TN_STORE_ID' estén configurados.")
    SHPAT = None
    STORE_ID = None

# Headers globales para todas las funciones
headers = {
    "Authentication": f"bearer {SHPAT}",
    "Content-Type": "application/json",
    "User-Agent": "CreacionesInfinitas" # Se eliminó el correo electrónico para mayor privacidad y profesionalismo
}

# URL base, utilizando el STORE_ID seguro
url_base = f"https://api.tiendanube.com/2025-03/{STORE_ID}/products" if STORE_ID else ""

## 🌐 Verificación de Conexión a la API
Esta función (`verificar_conexion()`) realiza una solicitud GET simple a la API de Tienda Nube para confirmar que las credenciales son válidas y que existe una conexión exitosa con la tienda configurada. Reporta cualquier error de conexión.

In [ ]:
def verificar_conexion():
    if not url_base or not SHPAT or not STORE_ID:
        print("⚠️ Configuración incompleta: Asegúrate de que SHPAT y STORE_ID estén cargados.")
        return
    try:
        res = requests.get(url_base, headers=headers)
        if res.status_code == 200:
            print(f"✅ Conexión exitosa con la tienda {STORE_ID}")
        else:
            print(f"❌ Error de conexión: {res.status_code} - {res.text}")
    except requests.exceptions.RequestException as e:
        print(f"❌ Error de red o conexión: {e}")

verificar_conexion()

## 🗑️ Función para Borrar Todo el Catálogo
La función `borrar_todo_el_catalogo()` está diseñada para eliminar masivamente todos los productos de la tienda. Incluye manejo de errores robusto y utiliza `tqdm` para mostrar el progreso de la operación. **¡Úsala con precaución, ya que la acción es irreversible!**

In [ ]:
def borrar_todo_el_catalogo():
    if not url_base or not SHPAT or not STORE_ID:
        print("⚠️ Configuración incompleta: Asegúrate de que SHPAT y STORE_ID estén cargados.")
        return

    try:
        # Primero obtenemos la lista de productos
        productos_response = requests.get(f"{url_base}?per_page=200", headers=headers)
        productos_response.raise_for_status() # Raise an exception for HTTP errors
        productos = productos_response.json()

        if not isinstance(productos, list):
            print("❌ Respuesta inesperada de la API. No se pudo obtener la lista de productos.")
            return

        if not productos:
            print("✅ La tienda ya está vacía, no hay productos para borrar.")
            return

        print(f"Iniciando borrado masivo de {len(productos)} productos...")
        total_borrados = 0
        # El tqdm va acá, envolviendo la lista de productos
        for p in tqdm(productos, desc="Borrando productos"): # Added tqdm
            if 'id' in p:
                delete_url = f"{url_base}/{p['id']}"
                delete_response = requests.delete(delete_url, headers=headers)
                delete_response.raise_for_status() # Raise for HTTP errors
                total_borrados += 1
                # print(f"🗑️ Borrado: {p.get('name', {}).get('es', 'Producto sin nombre')}") # Tqdm handles progress
            else:
                print(f"⚠️ Producto sin ID, saltando: {p}")
            time.sleep(0.5)

        print(f"\n✨ ¡Borrado completado! Se eliminaron {total_borrados} productos.")

    except requests.exceptions.HTTPError as e:
        print(f"❌ Error HTTP durante el borrado: {e.response.status_code} - {e.response.text}")
        print("Asegúrate de que tu SHPAT sea válido y no haya expirado.")
    except requests.exceptions.RequestException as e:
        print(f"❌ Error de conexión o red: {e}")
    except requests.exceptions.JSONDecodeError:
        print("❌ Error: La respuesta de la API no es un JSON válido.")

# Solo la llamás cuando realmente querés borrar
# borrar_todo_el_catalogo()
